In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE

C:\Users\Jay Patel\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
fs = pd.read_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\pp\preprocessed_properties.csv")
fs

,property_type,society,sector,price_in_cr,total_area_sqft,bedrooms,bathrooms,balcony,floornum_category,study_room,...,facing,furnishing_type,age_possession,super_built_up_area_missing,built_up_area_missing,carpet_area_missing,plot_area_missing,area_per_bedroom,dense_house_flag,luxury_category
0,Independent House,SS Linden Floors,sector 84,2.76,1800.0,4,4,2,Low-rise,0,...,north-east,Unfurnished,New Property,1,1,0,1,450.000000,0,Luxury
1,Independent Builder Floor,"Block E DLF City Phase 1, Gurgaon",sector 26,4.25,2700.0,3,3,2,Low-rise,0,...,not available,Unfurnished,New Property,1,1,0,1,900.000000,0,Budget
2,Independent Builder Floor,"Sushant Lok Phase 1, Gurgaon",sector 43,5.25,4000.0,4,4,2,Low-rise,1,...,east,Furnished,Relatively New,1,1,0,1,1000.000000,0,Budget
3,Independent House,International City by SOBHA Phase 2,sector 109,9.00,4500.0,4,5,3+,Low-rise,0,...,west,Unfurnished,Relatively New,1,1,1,0,1125.000000,0,Budget
4,Flat,godrej meridien,sector 106,3.11,2002.0,3,3,3,Low-rise,0,...,east,Unfurnished,New Property,1,0,1,1,667.333333,0,Budget
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5526,Independent House,Central Park Flower Valley Fleur Villas,sector 33,7.15,2250.0,4,4,3+,Low-rise,1,...,north-west,Semi-Furnished,Relatively New,1,0,0,0,562.500000,0,Luxury
5527,Independent Builder Floor,Whiteland Blissville,sector 76,2.15,1656.0,3,2,2,Low-rise,0,...,north-east,Unfurnished,New Property,0,1,1,1,552.000000,0,Luxury
5528,Independent Builder Floor,Signature Global Park,sector 33,0.95,1024.0,2,2,3,Low-rise,0,...,north-east,Unfurnished,New Property,0,1,1,1,512.000000,0,Luxury
5529,Independent Builder Floor,Huda Flats,sector 57,3.50,2650.0,4,4,2,Low-rise,0,...,east,Semi-Furnished,New Property,0,1,1,1,662.500000,0,Luxury


In [3]:
fs.isnull().sum()

property_type                  0
society                        0
sector                         0
price_in_cr                    0
total_area_sqft                0
bedrooms                       0
bathrooms                      0
balcony                        0
floornum_category              0
study_room                     0
servant_room                   0
store_room                     0
pooja_room                     0
others                         0
facing                         0
furnishing_type                0
age_possession                 0
super_built_up_area_missing    0
built_up_area_missing          0
carpet_area_missing            0
plot_area_missing              0
area_per_bedroom               0
dense_house_flag               0
luxury_category                0
dtype: int64

In [4]:
corr = fs.corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=True,          
    color_continuous_scale='RdBu_r',  
    title="Correlation Heatmap"
)

fig.update_layout(
    xaxis_title="Features",
    yaxis_title="Features",
    width=800,
    height=700
)

In [5]:
corr['price_in_cr'].sort_values(ascending= False)

price_in_cr                    1.000000
total_area_sqft                0.717601
bathrooms                      0.618032
bedrooms                       0.558544
area_per_bedroom               0.376666
super_built_up_area_missing    0.362468
servant_room                   0.354156
store_room                     0.319872
pooja_room                     0.319040
study_room                     0.302095
others                         0.028884
carpet_area_missing            0.014941
dense_house_flag              -0.037302
built_up_area_missing         -0.063005
plot_area_missing             -0.492666
Name: price_in_cr, dtype: float64

In [8]:
df = fs.copy()

###  Step 1: Separate Features and Target

In [10]:
X = df.drop(columns=["price_in_cr"])
y = np.log1p(df["price_in_cr"])   


print("Full dataset shape:", X.shape)

Full dataset shape: (5531, 23)


##### Split FIRST

In [11]:
X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape used for feature ranking:", X_train.shape)

Training shape used for feature ranking: (4424, 23)


### Step 2: Encode Categorical Features (For Tree Models)

In [13]:
X_train_enc = X_train.copy()
y_train_enc = np.log1p(y_train)

categorical_cols = X_train_enc.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:", categorical_cols)

if len(categorical_cols) > 0:
    encoder = OrdinalEncoder()
    X_train_enc[categorical_cols] = encoder.fit_transform(
        X_train_enc[categorical_cols]
    )

print("Encoded training shape:", X_train_enc.shape)

Categorical columns: ['property_type', 'society', 'sector', 'balcony', 'floornum_category', 'facing', 'furnishing_type', 'age_possession', 'luxury_category']
Encoded training shape: (4424, 23)


### Step 3: Random Forest Feature Importance

In [15]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_enc, y_train_enc)

rf_imp = pd.Series(
    rf.feature_importances_,
    index=X_train_enc.columns,
    name="rf"
)

rf_imp.sort_values(ascending=False).head(20)

total_area_sqft          0.612287
property_type            0.147382
sector                   0.070899
society                  0.041195
plot_area_missing        0.034250
area_per_bedroom         0.025285
bathrooms                0.014781
balcony                  0.008888
servant_room             0.006440
facing                   0.006207
bedrooms                 0.006017
age_possession           0.005471
luxury_category          0.004821
furnishing_type          0.004554
pooja_room               0.002263
study_room               0.001773
store_room               0.001432
floornum_category        0.001413
built_up_area_missing    0.001269
others                   0.001254
Name: rf, dtype: float64

### Step 4: Gradient Boosting Importance

In [16]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_enc, y_train_enc)

gb_imp = pd.Series(
    gb.feature_importances_,
    index=X_train_enc.columns,
    name="gb"
)

gb_imp.sort_values(ascending=False).head()

total_area_sqft      0.566086
property_type        0.128469
bathrooms            0.099996
sector               0.075181
plot_area_missing    0.042858
Name: gb, dtype: float64

###  Step 5: Permutation Importance

In [17]:
perm = permutation_importance(
    rf,
    X_train_enc,
    y_train_enc,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_imp = pd.Series(
    perm.importances_mean,
    index=X_train_enc.columns,
    name="perm"
)

perm_imp.sort_values(ascending=False).head()

total_area_sqft      0.956428
property_type        0.217347
sector               0.163394
society              0.073392
plot_area_missing    0.064383
Name: perm, dtype: float64

### Step 6: REF

In [18]:
rfe = RFE(
    RandomForestRegressor(n_estimators=200, random_state=42),
    n_features_to_select= 16
)

rfe.fit(X_train_enc, y_train_enc)

rfe_rank = pd.Series(
    rfe.ranking_,
    index=X_train_enc.columns,
    name="rfe_rank"
)

rfe_rank.sort_values().head(20)

property_type          1
area_per_bedroom       1
plot_area_missing      1
age_possession         1
furnishing_type        1
facing                 1
servant_room           1
study_room             1
pooja_room             1
balcony                1
bathrooms              1
bedrooms               1
total_area_sqft        1
sector                 1
society                1
luxury_category        1
carpet_area_missing    2
floornum_category      3
store_room             4
others                 5
Name: rfe_rank, dtype: int32

### Step 7: Combine All Importance Scores

In [19]:
importance_df = pd.concat([rf_imp, gb_imp, perm_imp], axis=1)

# Normalize columns
importance_df = (importance_df - importance_df.min()) / (
    importance_df.max() - importance_df.min()
)

importance_df["avg_importance"] = importance_df.mean(axis=1)

# Convert RFE ranking into score
importance_df["rfe_score"] = 1 / rfe_rank

importance_df["final_score"] = (
    importance_df["avg_importance"] + importance_df["rfe_score"]
) / 2

importance_df = importance_df.sort_values("final_score", ascending=False)

importance_df.head(20)

,rf,gb,perm,avg_importance,rfe_score,final_score
total_area_sqft,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
property_type,0.240282,0.226943,0.227083,0.231436,1.000000,0.615718
sector,0.115297,0.132809,0.170660,0.139589,1.000000,0.569794
bathrooms,0.023594,0.176645,0.029687,0.076642,1.000000,0.538321
plot_area_missing,0.055409,0.075709,0.067117,0.066078,1.000000,0.533039
society,0.066757,0.044992,0.076537,0.062762,1.000000,0.531381
area_per_bedroom,0.040759,0.022105,0.040479,0.034448,1.000000,0.517224
bedrooms,0.009271,0.031950,0.010108,0.017110,1.000000,0.508555
servant_room,0.009962,0.024185,0.014126,0.016091,1.000000,0.508046
balcony,0.013964,0.010267,0.011376,0.011869,1.000000,0.505934


### Step 8: Final Top 16 

In [27]:
selected_features = importance_df.head(15).index.tolist()

print("Selected Top Features:")
for f in selected_features:
    print("-", f)

Selected Top Features:
- total_area_sqft
- property_type
- sector
- bathrooms
- plot_area_missing
- society
- area_per_bedroom
- bedrooms
- servant_room
- balcony
- furnishing_type
- luxury_category
- facing
- age_possession
- pooja_room


In [25]:
fs_df = df[selected_features + ['price_in_cr']].copy()

print("Final shape after feature selection:", fs_df.shape)
fs_df.head()

Final shape after feature selection: (5531, 16)


,total_area_sqft,property_type,sector,bathrooms,plot_area_missing,society,area_per_bedroom,bedrooms,servant_room,balcony,furnishing_type,luxury_category,facing,age_possession,pooja_room,price_in_cr
0,1800.0,Independent House,sector 84,4,1,SS Linden Floors,450.000000,4,0,2,Unfurnished,Luxury,north-east,New Property,0,2.76
1,2700.0,Independent Builder Floor,sector 26,3,1,"Block E DLF City Phase 1, Gurgaon",900.000000,3,0,2,Unfurnished,Budget,not available,New Property,0,4.25
2,4000.0,Independent Builder Floor,sector 43,4,1,"Sushant Lok Phase 1, Gurgaon",1000.000000,4,1,2,Furnished,Budget,east,Relatively New,1,5.25
3,4500.0,Independent House,sector 109,5,0,International City by SOBHA Phase 2,1125.000000,4,1,3+,Unfurnished,Budget,west,Relatively New,0,9.00
4,2002.0,Flat,sector 106,3,1,godrej meridien,667.333333,3,1,3,Unfurnished,Budget,east,New Property,0,3.11
